In [1]:
import os
import glob
import pickle
import pandas as pd
import numpy as np

from dask.diagnostics import ProgressBar

from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell
import scanpy as sc
import seaborn as sns
from pyscenic.utils import load_motif_annotations
from ctxcore.recovery import aucs as calc_aucs

In [9]:
def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

[FeatherRankingDatabase(name="concat_arid3a_supp.genes_vs_tracks.rankings"),
 FeatherRankingDatabase(name="concat_arid3a_glue.genes_vs_tracks.rankings")]

In [6]:
rna = sc.read("../process/scglue/rna_temp.h5ad")
ex_matrix = pd.DataFrame(rna.layers["counts"].toarray())
ex_matrix.index  = rna.obs_names
ex_matrix.columns  = rna.var_names

In [7]:
adjacencies = pd.read_csv("../process/Arid3a/scenic/8.25_vargene_draft_grn.csv")

In [8]:
modules = list(modules_from_adjacencies(adjacencies, ex_matrix))


2024-08-28 10:58:43,558 - pyscenic.utils - INFO - Calculating Pearson correlations.

2024-08-28 10:58:43,572 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].

2024-08-28 10:58:44,913 - pyscenic.utils - INFO - Creating modules.


In [79]:
MOTIF_ANNOTATIONS_FNAME = "../process/Arid3a/glue_regulon/concat_arid3a_ctx_annotation.tsv"

In [78]:
DATABASES_GLOB="../process/Arid3a/ranking_feather/concat_arid3a_*.genes_vs_tracks.rankings.feather"
db_fnames = glob.glob(DATABASES_GLOB)
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

[FeatherRankingDatabase(name="concat_arid3a_supp.genes_vs_tracks.rankings"),
 FeatherRankingDatabase(name="concat_arid3a_glue.genes_vs_tracks.rankings")]

In [82]:
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df = prune2df(dbs, modules, MOTIF_ANNOTATIONS_FNAME,num_workers=20,nes_threshold=0,auc_threshold=0.5,rank_threshold=1800)

[########################################] | 100% Completed | 23.83 s


In [83]:
df

Enrichment                                  \
                            AUC       NES MotifSimilarityQvalue   
TF     MotifID                                                    
Arid3a Arid3a_1_glue   0.259584  1.324343                   0.0   
       Arid3a_5_glue   0.231995  0.920740                   0.0   
       Arid3a_1_glue   0.286621  1.495310                   0.0   
       Arid3a_5_glue   0.230360  0.675123                   0.0   
       Arid3a_1_glue   0.228400  0.694560                   0.0   
       Arid3a_3_glue   0.194540  0.214957                   0.0   
       Arid3a_5_glue   0.271640  1.307023                   0.0   
       Arid3a_1_glue   0.255076  1.205529                   0.0   
       Arid3a_5_glue   0.242694  1.048182                   0.0   
       Arid3a_1_glue   0.252535  1.203526                   0.0   
       Arid3a_5_glue   0.239748  1.036741                   0.0   
       Arid3a_1_glue   0.252342  1.178749                   0.0   
       Arid3a_5_glue   0.243286  1.060646                   0.0   
       Arid3a_1_supp   0.182428  1.251023                   0.0   
       Arid3a_3_supp   0.114936  0.286151                   0.0   
       Arid3a_5_supp   0.150229  0.790702                   0.0   
       Arid3a_1_supp   0.185239  1.354707                   0.0   
       Arid3a_3_supp   0.112849  0.295460                   0.0   
       Arid3a_5_supp   0.136309  0.638733                   0.0   
       Arid3a_1_supp   0.111620  1.641987                   0.0   
       Arid3a_5_supp   0.064480  0.526409                   0.0   
       Arid3a_1_supp   0.201726  1.229146                   0.0   
       Arid3a_3_supp   0.125771  0.280825                   0.0   
       Arid3a_5_supp   0.169336  0.824748                   0.0   
       Arid3a_1_supp   0.193146  1.200122                   0.0   
       Arid3a_3_supp   0.122803  0.282915                   0.0   
       Arid3a_5_supp   0.166905  0.857957                   0.0   
       Arid3a_1_supp   0.195966  1.197986                   0.0   
       Arid3a_3_supp   0.126612  0.306696                   0.0   
       Arid3a_5_supp   0.168337  0.842916                   0.0   

                                                       \
                     OrthologousIdentity   Annotation   
TF     MotifID                                          
Arid3a Arid3a_1_glue                 1.0  placeholder   
       Arid3a_5_glue                 1.0  placeholder   
       Arid3a_1_glue                 1.0  placeholder   
       Arid3a_5_glue                 1.0  placeholder   
       Arid3a_1_glue                 1.0  placeholder   
       Arid3a_3_glue                 1.0  placeholder   
       Arid3a_5_glue                 1.0  placeholder   
       Arid3a_1_glue                 1.0  placeholder   
       Arid3a_5_glue                 1.0  placeholder   
       Arid3a_1_glue                 1.0  placeholder   
       Arid3a_5_glue                 1.0  placeholder   
       Arid3a_1_glue                 1.0  placeholder   
       Arid3a_5_glue                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  

In [13]:
df

Enrichment                                  \
                            AUC       NES MotifSimilarityQvalue   
TF     MotifID                                                    
Arid3a Arid3a_1_supp   0.032311  1.576428                   0.0   
       Arid3a_2_supp   0.024897  0.177845                   0.0   
       Arid3a_4_supp   0.025927  0.372092                   0.0   
       Arid3a_1_supp   0.039301  1.474819                   0.0   
       Arid3a_2_supp   0.029301  0.543643                   0.0   
       Arid3a_4_supp   0.025441  0.184181                   0.0   
       Arid3a_1_supp   0.032800  0.784370                   0.0   
       Arid3a_4_supp   0.032800  0.784370                   0.0   
       Arid3a_5_supp   0.031800  0.661812                   0.0   
       Arid3a_2_supp   0.023857  1.147999                   0.0   
       Arid3a_3_supp   0.023757  1.086411                   0.0   
       Arid3a_1_supp   0.024211  0.703387                   0.0   
       Arid3a_2_supp   0.024523  0.853548                   0.0   
       Arid3a_3_supp   0.024589  0.885161                   0.0   
       Arid3a_1_supp   0.025394  0.982042                   0.0   
       Arid3a_2_supp   0.024513  0.531346                   0.0   
       Arid3a_3_supp   0.025131  0.847624                   0.0   

                                                       \
                     OrthologousIdentity   Annotation   
TF     MotifID                                          
Arid3a Arid3a_1_supp                 1.0  placeholder   
       Arid3a_2_supp                 1.0  placeholder   
       Arid3a_4_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_2_supp                 1.0  placeholder   
       Arid3a_4_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_4_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_2_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_2_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_2_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   

                                                                         \
                                                                Context   
TF     MotifID                                                            
Arid3a Arid3a_1_supp  frozenset({'weight>75.0%', 'activating', 'conc...   
       Arid3a_2_supp  frozenset({'weight>75.0%', 'activating', 'conc...   
       Arid3a_4_supp  frozenset({'weight>75.0%', 'activating', 'conc...   
       Arid3a_1_supp  frozenset({'weight>90.0%', 'activating', 'conc...   
       Arid3a_2_supp  frozenset({'weight>90.0%', 'activating', 'conc...   
       Arid3a_4_supp  frozenset({'weight>90.0%', 'activating', 'conc...   
       Arid3a_1_supp  frozenset({'top50', 'activating', 'concat_arid...   
       Arid3a_4_supp  frozenset({'top50', 'activating', 'concat_arid...   
       Arid3a_5_supp  frozenset({'top50', 'activating', 'concat_arid...   
       Arid3a_2_supp  frozenset({'top5perTarget', 'activating', 'con...   
       Arid3a_3_supp  frozenset({'top5perTarget', 'activating', 'con...   
       Arid3a_1_supp  frozenset({'activating', 'top10perTarget', 'co...   
       Arid3a_2_supp  frozenset({'activating', 'top10perTarget', 'co...   
       Arid3a_3_supp  frozenset({'activating', 'top10perTarget', 'co...   
       Arid3a_1_supp  frozenset({'activating', 'concat_arid3a_supp.g...   
       Arid3a_2_supp  frozenset({'activating', 'concat_arid3a_supp.g...   
       Arid3a_3_supp  frozenset({'activating', 'concat_arid3a_supp.g...   

                                      

In [74]:
DATABASES_GLOB="../process/Arid3a/ranking_feather/feather_test_concat.genes_vs_tracks.rankings.feather"
db_fnames = glob.glob(DATABASES_GLOB)
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

[FeatherRankingDatabase(name="feather_test_concat.genes_vs_tracks.rankings")]

In [75]:
MOTIF_ANNOTATIONS_FNAME = "../process/Arid3a/glue_regulon/test_var_ctx_annotation.tsv"

In [88]:
MOTIF_ANNOTATIONS_FNAME = "../process/Arid3a/glue_regulon/concat_arid3a_var_ctx_annotation.tsv"

In [77]:
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df = prune2df(dbs, modules, MOTIF_ANNOTATIONS_FNAME,num_workers=20,nes_threshold=0,,auc_threshold=0.5,rank_threshold=1800)

[                                        ] | 0% Completed | 64.30 sms



KeyboardInterrupt



In [ ]:
df

In [56]:
db_fnames

['../process/Arid3a/ranking_feather/feather_test_concat.genes_vs_tracks.rankings.feather']

In [50]:
dbs

[FeatherRankingDatabase(name="feather_test_concat.genes_vs_tracks.rankings")]

In [71]:
DATABASES_GLOB="../process/Arid3a/ranking_feather/input*.genes_vs_tracks.rankings.feather"
db_fnames = glob.glob(DATABASES_GLOB)
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df2 = prune2df(dbs, modules, MOTIF_ANNOTATIONS_FNAME,num_workers=20,nes_threshold=0,auc_threshold=0.5,rank_threshold=1800)

[########################################] | 100% Completed | 27.16 s


In [72]:
df2.shape

(27, 8)

In [73]:
df2

Enrichment                                  \
                            AUC       NES MotifSimilarityQvalue   
TF     MotifID                                                    
Arid3a Arid3a_1_supp   0.182428  0.725764                   0.0   
       Arid3a_5_supp   0.150229  0.282923                   0.0   
Ascl1  Ascl1_glue      0.319486  2.525084                   0.0   
Batf   Batf_glue       0.200887  1.349545                   0.0   
Arid3a Arid3a_1_supp   0.185239  0.568507                   0.0   
Ascl1  Ascl1_glue      0.305893  1.991724                   0.0   
Batf   Batf_glue       0.204905  1.204473                   0.0   
Ascl1  Ascl1_glue      0.316128  2.546867                   0.0   
Batf   Batf_glue       0.199137  1.141475                   0.0   
Arid3a Arid3a_1_supp   0.201726  1.279236                   0.0   
       Arid3a_3_supp   0.125771  0.176232                   0.0   
       Arid3a_5_supp   0.169336  0.808875                   0.0   
Arntl  Arntl_glue      0.141538  0.275927                   0.0   
Ascl1  Ascl1_glue      0.343261  2.681933                   0.0   
Batf   Batf_glue       0.097071  0.271884                   0.0   
Arid3a Arid3a_1_supp   0.193146  1.086762                   0.0   
       Arid3a_3_supp   0.122803  0.071200                   0.0   
       Arid3a_5_supp   0.166905  0.707906                   0.0   
Arntl  Arntl_glue      0.151862  0.058070                   0.0   
Ascl1  Ascl1_glue      0.338927  2.689662                   0.0   
Batf   Batf_glue       0.169432  1.020906                   0.0   
Arid3a Arid3a_1_supp   0.195966  1.093780                   0.0   
       Arid3a_3_supp   0.126612  0.101699                   0.0   
       Arid3a_5_supp   0.168337  0.698557                   0.0   
Arntl  Arntl_glue      0.148095  0.029302                   0.0   
Ascl1  Ascl1_glue      0.316128  2.546867                   0.0   
Batf   Batf_glue       0.158685  0.978586                   0.0   

                                                       \
                     OrthologousIdentity   Annotation   
TF     MotifID                                          
Arid3a Arid3a_1_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
Ascl1  Ascl1_glue                    1.0  placeholder   
Batf   Batf_glue                     1.0  placeholder   
Arid3a Arid3a_1_supp                 1.0  placeholder   
Ascl1  Ascl1_glue                    1.0  placeholder   
Batf   Batf_glue                     1.0  placeholder   
Ascl1  Ascl1_glue                    1.0  placeholder   
Batf   Batf_glue                     1.0  placeholder   
Arid3a Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
Arntl  Arntl_glue                    1.0  placeholder   
Ascl1  Ascl1_glue                    1.0  placeholder   
Batf   Batf_glue                     1.0  placeholder   
Arid3a Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
Arntl  Arntl_glue                    1.0  placeholder   
Ascl1  Ascl1_glue                    1.0  placeholder   
Batf   Batf_glue                     1.0  placeholder   
Arid3a Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
Arntl  Arntl_glue                    1.0  placeholder   
Ascl1  Ascl1_glue                    1.0  placeholder   
Batf   Batf_glue                     1.0  placeholder   

                                                                         \
                                                                Context   
TF     MotifID                                                            
Arid3a Arid3a_1_supp  frozenset({'weight>75.0%', 'activating', 'feat...   

In [60]:
df = pd.read_table(MOTIF_ANNOTATIONS_FNAME)

In [96]:
DATABASES_GLOB="../process/Arid3a/ranking_feather/input*.genes_vs_tracks.rankings.feather"
db_fnames = glob.glob(DATABASES_GLOB)
dbs_short = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs_short


[FeatherRankingDatabase(name="input_glue.genes_vs_tracks.rankings"),
 FeatherRankingDatabase(name="input_feather_test_short.genes_vs_tracks.rankings")]

In [97]:
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df_short = prune2df(dbs_short, modules, MOTIF_ANNOTATIONS_FNAME,num_workers=20,nes_threshold=0,auc_threshold=0.5)

[########################################] | 100% Completed | 33.64 s


In [99]:
!mkdir ../result/Arid3a/regulon

In [101]:

df_short.to_csv("../result/Arid3a/regulon/8.25_regulon_short.csv")

In [102]:
regulons = df2regulons(df_short)

Create regulons from a dataframe of enriched features.
Additional columns saved: []


In [105]:
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df_short2 = prune2df(dbs_short, modules, MOTIF_ANNOTATIONS_FNAME,num_workers=20,nes_threshold=0,auc_threshold=0.8,rank_threshold=1800)

[########################################] | 100% Completed | 36.57 s


In [106]:
df_short2

Enrichment                                  \
                             AUC       NES MotifSimilarityQvalue   
TF      MotifID                                                    
Arid3a  Arid3a_1_glue   0.418426  1.497857                   0.0   
Ascl1   Ascl1_glue      0.463764  2.314943                   0.0   
Batf    Batf_glue       0.395932  1.223696                   0.0   
Arid3a  Arid3a_1_glue   0.444637  1.806761                   0.0   
Ascl1   Ascl1_glue      0.464687  1.723943                   0.0   
Batf    Batf_glue       0.400107  0.804214                   0.0   
Arid3a  Arid3a_1_glue   0.414566  0.660410                   0.0   
        Arid3a_3_glue   0.410468  0.536763                   0.0   
        Arid3a_4_glue   0.421437  0.867746                   0.0   
        Arid3a_5_glue   0.430081  1.128612                   0.0   
Ascl1   Ascl1_glue      0.458424  2.474658                   0.0   
Batf    Batf_glue       0.395525  0.764042                   0.0   
Arid3a  Arid3a_1_glue   0.412929  2.719785                   0.0   
        Arid3a_5_glue   0.391587  0.755644                   0.0   
Arntl   Arntl_glue      0.407774  0.662028                   0.0   
Ascl1   Ascl1_glue      0.466122  2.266564                   0.0   
Arid3a  Arid3a_1_glue   0.409847  2.274908                   0.0   
        Arid3a_5_glue   0.388763  0.204798                   0.0   
Arntl   Arntl_glue      0.415306  0.137774                   0.0   
Ascl1   Ascl1_glue      0.479045  2.842003                   0.0   
Batf    Batf_glue       0.369668  0.316010                   0.0   
Creb3l1 Creb3l1_glue    0.381186  0.114010                   0.0   
Arid3a  Arid3a_1_glue   0.410127  2.170185                   0.0   
        Arid3a_5_glue   0.392016  0.395151                   0.0   
Arntl   Arntl_glue      0.412799  0.070556                   0.0   
Ascl1   Ascl1_glue      0.458424  2.474658                   0.0   
        Ascl1_glue      0.463764  2.308760                   0.0   
Batf    Batf_glue       0.395932  1.240646                   0.0   
Ascl1   Ascl1_glue      0.464687  1.743433                   0.0   
Batf    Batf_glue       0.400107  0.802305                   0.0   
Ascl1   Ascl1_glue      0.458424  2.490899                   0.0   
Batf    Batf_glue       0.395525  0.724467                   0.0   
Arid3a  Arid3a_3_supp   0.389724  0.673378                   0.0   
        Arid3a_5_supp   0.390716  0.762273                   0.0   
Arntl   Arntl_glue      0.407774  0.697083                   0.0   
Ascl1   Ascl1_glue      0.466122  2.484245                   0.0   
Arid3a  Arid3a_3_supp   0.387620  0.196756                   0.0   
        Arid3a_5_supp   0.388837  0.302415                   0.0   
Arntl   Arntl_glue      0.415306  0.217510                   0.0   
Ascl1   Ascl1_glue      0.479045  2.874988                   0.0   
Batf    Batf_glue       0.369668  0.218634                   0.0   
Creb3l1 Creb3l1_glue    0.381186  0.137019                   0.0   
Arid3a  Arid3a_3_supp   0.390294  0.316241                   0.0   
        Arid3a_5_supp   0.389905  0.282530                   0.0   
Arntl   Arntl_glue      0.412799  0.155928                   0.0   
Ascl1   Ascl1_glue      0.458424  2.490899                   0.0   

                                                        \
                      OrthologousIdentity   Annotation   
TF      MotifID                                          
Arid3a  Arid3a_1_glue                 1.0  placeholder   
Ascl1   Ascl1_glue                    1.0  placeholder   
Batf    Batf_glue                     1.0  placeholder   
Arid3a  Arid3a_1_glue                 1.0  placeholder   
Ascl1   Ascl1_glue                    1.0  placeholder   
Batf    Batf_glue                     1.0  placeholder   
Arid3a  Arid3a_1_glue                 1.0  placeholder   
        Arid3a_3_glue                 1.0  placeholder   
        Arid3a_4_glue                 1.0  placehol

In [95]:
test

,0610038B21Rik,1010001B22Rik,1110002O04Rik,1110006O24Rik,1110035H17Rik,1110046J04Rik,1500002C15Rik,1500026H17Rik,1600002D24Rik,1700001L05Rik,...,Zic4,Zkscan2,Zkscan7,Zmat1,Zmat4,Zpbp,Zscan2,Zswim1,Zswim5,tracks
0,1740,689,777,1740,358,1058,1740,1052,1381,890,...,768,1740,370,273,78,654,992,1740,266,Arid3a_1_glue
1,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,...,1085,1085,1085,1085,121,1085,1085,1085,1085,Arid3a_2_glue
2,1186,130,1186,1186,1186,1186,1186,1186,337,1186,...,1186,1186,1186,1186,1186,1186,1186,39,1186,Arid3a_3_glue
3,1063,1063,1063,1063,1063,1063,1063,1063,1063,1063,...,1063,1063,1063,1063,1063,1063,1063,1063,1063,Arid3a_4_glue
4,299,271,46,378,1353,1353,1353,1353,668,1353,...,1353,1353,1353,1353,648,1353,290,247,229,Arid3a_5_glue


In [90]:
df_short

Enrichment                                  \
                            AUC       NES MotifSimilarityQvalue   
TF     MotifID                                                    
Arid3a Arid3a_1_supp   0.182428  1.251023                   0.0   
       Arid3a_3_supp   0.114936  0.286151                   0.0   
       Arid3a_5_supp   0.150229  0.790702                   0.0   
       Arid3a_1_supp   0.185239  1.354707                   0.0   
       Arid3a_3_supp   0.112849  0.295460                   0.0   
       Arid3a_5_supp   0.136309  0.638733                   0.0   
       Arid3a_1_supp   0.111620  1.641987                   0.0   
       Arid3a_5_supp   0.064480  0.526409                   0.0   
       Arid3a_1_supp   0.201726  1.229146                   0.0   
       Arid3a_3_supp   0.125771  0.280825                   0.0   
       Arid3a_5_supp   0.169336  0.824748                   0.0   
       Arid3a_1_supp   0.193146  1.200122                   0.0   
       Arid3a_3_supp   0.122803  0.282915                   0.0   
       Arid3a_5_supp   0.166905  0.857957                   0.0   
       Arid3a_1_supp   0.195966  1.197986                   0.0   
       Arid3a_3_supp   0.126612  0.306696                   0.0   
       Arid3a_5_supp   0.168337  0.842916                   0.0   
       Arid3a_1_supp   0.182428  0.725764                   0.0   
       Arid3a_5_supp   0.150229  0.282923                   0.0   
Ascl1  Ascl1_glue      0.319486  2.525084                   0.0   
Batf   Batf_glue       0.200887  1.349545                   0.0   
Arid3a Arid3a_1_supp   0.185239  0.568507                   0.0   
Ascl1  Ascl1_glue      0.305893  1.991724                   0.0   
Batf   Batf_glue       0.204905  1.204473                   0.0   
Ascl1  Ascl1_glue      0.316128  2.546867                   0.0   
Batf   Batf_glue       0.199137  1.141475                   0.0   
Arid3a Arid3a_1_supp   0.201726  1.279236                   0.0   
       Arid3a_3_supp   0.125771  0.176232                   0.0   
       Arid3a_5_supp   0.169336  0.808875                   0.0   
Arntl  Arntl_glue      0.141538  0.275927                   0.0   
Ascl1  Ascl1_glue      0.343261  2.681933                   0.0   
Batf   Batf_glue       0.097071  0.271884                   0.0   
Arid3a Arid3a_1_supp   0.193146  1.086762                   0.0   
       Arid3a_3_supp   0.122803  0.071200                   0.0   
       Arid3a_5_supp   0.166905  0.707906                   0.0   
Arntl  Arntl_glue      0.151862  0.058070                   0.0   
Ascl1  Ascl1_glue      0.338927  2.689662                   0.0   
Batf   Batf_glue       0.169432  1.020906                   0.0   
Arid3a Arid3a_1_supp   0.195966  1.093780                   0.0   
       Arid3a_3_supp   0.126612  0.101699                   0.0   
       Arid3a_5_supp   0.168337  0.698557                   0.0   
Arntl  Arntl_glue      0.148095  0.029302                   0.0   
Ascl1  Ascl1_glue      0.316128  2.546867                   0.0   
Batf   Batf_glue       0.158685  0.978586                   0.0   

                                                       \
                     OrthologousIdentity   Annotation   
TF     MotifID                                          
Arid3a Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_supp                 1.0  placeholder   
       Arid3a_5_supp                 1.0  placeholder   
       Arid3a_1_supp                 1.0  placeholder   
       Arid3a_3_